# Imports

In [1]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import json
import random
from torch.utils.data import DataLoader, Dataset
import torch.nn as nn
from datetime import datetime

import sys

ROOT = Path.cwd().resolve()
if not (ROOT / 'tap_recognition').is_dir():
    ROOT = ROOT.parent
assert (ROOT / 'tap_recognition').is_dir(), 'Start from the repo root or notebooks/'
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'notebooks'))


from tap_recognition.config import DataConfig, ModelConfig, \
    TrainConfig, TrainingConfig, LabelConfig, EarlyStoppingConfig
from tap_recognition.dataset import RecordedIMUDataset
from tap_recognition.model import CausalCNNGRU
from train import build_datasets, EarlyStopping


In [2]:
cfg = TrainConfig(out_dir='testing_checkpoints')

for key in ('train_dir', 'val_dir'):
    setattr(cfg.data, key, ','.join(str(ROOT / p.strip()) for p in getattr(cfg.data, key).split(',')))

print(json.dumps(cfg.to_dict(), indent=2))




{
  "seed": 42,
  "device": "auto",
  "out_dir": "testing_checkpoints",
  "init_checkpoint": null,
  "freeze_except_last": false,
  "data": {
    "mode": "recorded",
    "train_dir": "/home/aprohack/desktop/work/tap-project/project/tapRecognitionRui/TapRecognition/data/train_data",
    "val_dir": "/home/aprohack/desktop/work/tap-project/project/tapRecognitionRui/TapRecognition/data/valid_data",
    "window_samples": 300,
    "sample_rate": 100.0,
    "highpass": true,
    "negative_windows_per_file": 20,
    "trigger_context_samples": 200,
    "exclusion_margin": 200,
    "dt_min": 0.12,
    "dt_max": 0.45,
    "synthetic_samples": 8000,
    "val_split": 0.15
  },
  "labels": {
    "sigma": 0.04,
    "peak_offset": 0.05,
    "half_frames": 10
  },
  "model": {
    "input_dim": 6,
    "num_classes": 3,
    "cnn_channels": 32,
    "gru_hidden": 64,
    "gru_layers": 1,
    "kernel_size": 5,
    "dilations": [
      1,
      2,
      4
    ],
    "dropout": 0.1
  },
  "training": {
    "e

# Seed setting

In [3]:
def fix_seeds(seed: int):
    torch.manual_seed(seed)
    np.random.seed(seed=seed)
    random.seed(seed)
    torch.use_deterministic_algorithms(True)
fix_seeds(cfg.seed)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Train and Validation Datasets

In [4]:
train_ds, val_ds = build_datasets(cfg)
train_loader = DataLoader(
    dataset=train_ds,
    batch_size=cfg.training.batch_size,
    shuffle=True,
    num_workers=cfg.training.num_workers
)
val_loader = DataLoader(
    dataset=val_ds,
    batch_size=cfg.training.batch_size,
    shuffle=False,
    num_workers=cfg.training.num_workers
)

Train dirs:
  /home/aprohack/desktop/work/tap-project/project/tapRecognitionRui/TapRecognition/data/train_data
Val dirs:
  /home/aprohack/desktop/work/tap-project/project/tapRecognitionRui/TapRecognition/data/valid_data


# Model creation

In [5]:
model_kwargs = cfg.model.to_model_kwargs()
model_kwargs

{'input_dim': 6,
 'num_classes': 3,
 'cnn_channels': 32,
 'gru_hidden': 64,
 'gru_layers': 1,
 'kernel_size': 5,
 'dilations': (1, 2, 4),
 'dropout': 0.1}

In [6]:
assert (cfg.init_checkpoint is None), 'We are testing models training from scratch'

In [7]:
model = CausalCNNGRU(**model_kwargs).to(device)
assert not cfg.freeze_except_last, 'No fine tuning'

run_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")


out_dir = ROOT / Path(cfg.out_dir) / run_timestamp
out_dir.mkdir(parents=True, exist_ok=True)


In [8]:
trainable = list(model.parameters())

# Optimizer and Learning Rate

In [9]:
optimizer = torch.optim.AdamW(
    trainable,
    lr=cfg.training.lr,
    weight_decay=cfg.training.weight_decay,
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=cfg.training.epochs,
)

# Early Stopping and train loop configuration

In [10]:
best_val_acc = 0.0
early_stop = EarlyStopping(cfg.training.early_stopping)
stopped_epoch = cfg.training.epochs

In [11]:
cfg_es = cfg.training.early_stopping

## Positive and negative counts

In [12]:
train_ds[0]

{'imu': tensor([[-1.3999,  6.4064,  6.8494, -0.4611,  0.4121, -0.2047],
         [-1.1951,  6.2312,  5.7668, -0.3448,  0.3751, -0.2133],
         [-0.9145,  6.0428,  4.9876, -0.3011,  0.2092, -0.2427],
         ...,
         [ 6.8542,  1.5032, -5.8868, -2.5403,  2.0980,  0.8649],
         [ 6.7419,  1.5505, -4.2766, -3.9863,  3.7953,  0.5154],
         [ 6.7419,  1.5505, -4.2766, -3.9863,  3.7953,  0.5154]]),
 'frame_labels': tensor([[1., 0., 0.],
         [1., 0., 0.],
         [1., 0., 0.],
         [1., 0., 0.],
         [1., 0., 0.],
         [1., 0., 0.],
         [1., 0., 0.],
         [1., 0., 0.],
         [1., 0., 0.],
         [1., 0., 0.],
         [1., 0., 0.],
         [1., 0., 0.],
         [1., 0., 0.],
         [1., 0., 0.],
         [1., 0., 0.],
         [1., 0., 0.],
         [1., 0., 0.],
         [1., 0., 0.],
         [1., 0., 0.],
         [1., 0., 0.],
         [1., 0., 0.],
         [1., 0., 0.],
         [1., 0., 0.],
         [1., 0., 0.],
         [1., 0., 0

In [13]:
n_train_pos = 0
for i in range(len(train_ds)):
    # imu = train_ds[i]['imu']
    # label = train_ds[i]['frame_labels']
    window = train_ds[i]['window_label']
    if window > 0:
        n_train_pos += 1

n_val_pos = 0
for i in range(len(val_ds)):
    window = val_ds[i]['window_label']
    if window > 0:
        n_val_pos += 1

print(
    f"Training on {device}, {len(train_ds)} train "
    f"({n_train_pos} pos / {len(train_ds) - n_train_pos} neg) / "
    f"{len(val_ds)} val ({n_val_pos} pos / {len(val_ds) - n_val_pos} neg)"
)

print(
    f"Sequence length: {cfg.data.window_samples} frames "
    f"({cfg.data.window_samples / cfg.data.sample_rate:.2f} s)"
)
print(f"Model receptive field: {model.receptive_field} samples")
if cfg_es.enabled:
    print(
        f"Early stopping: monitor={cfg_es.monitor}, patience={cfg_es.patience}, "
        f"min_delta={cfg_es.min_delta}"
    )

Training on cpu, 700 train (440 pos / 260 neg) / 220 val (80 pos / 140 neg)
Sequence length: 300 frames (3.00 s)
Model receptive field: 29 samples
Early stopping: monitor=val_window_acc, patience=10, min_delta=0.001


# Train loop

In [14]:
from train import _augment_positive_windows, frame_soft_ce, window_class_from_probs

## Evaluation

In [15]:
@torch.no_grad()
def evaluate(
    model: CausalCNNGRU,
    loader: DataLoader,
    device: torch.device,
    threshold: float,
    event_weight: float = 10.0,
    neg_window_weight: float = 2.0,
) -> dict[str, float]:
    model.eval()
    total_loss = 0.0
    correct_frames = 0
    total_frames = 0
    correct_windows = 0
    total_windows = 0
    tp = fp = fn = tn = 0

    for batch in loader:
        imu = batch["imu"].to(device)
        frame_labels = batch["frame_labels"].to(device)
        window_labels = batch["window_label"].to(device)

        logits, _ = model(imu)
        probs = torch.softmax(logits, dim=-1)
        loss = frame_soft_ce(
            logits,
            frame_labels,
            event_weight,
            window_label=window_labels,
            neg_window_weight=neg_window_weight,
        )
        total_loss += loss.item() * imu.size(0)

        pred = probs.argmax(dim=-1)
        true_cls = frame_labels.argmax(dim=-1)
        correct_frames += (pred == true_cls).sum().item()
        total_frames += true_cls.numel()

        window_pred = window_class_from_probs(probs, threshold)
        correct_windows += (window_pred == window_labels).sum().item()
        total_windows += imu.size(0)

        pos_pred = (window_pred > 0).long()
        pos_true = (window_labels > 0).long()
        for wp, wl in zip(pos_pred, pos_true):
            if wp == 1 and wl == 1:
                tp += 1
            elif wp == 1 and wl == 0:
                fp += 1
            elif wp == 0 and wl == 1:
                fn += 1
            else:
                tn += 1

    n = len(loader.dataset)
    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    return {
        "loss": total_loss / n,
        "frame_acc": correct_frames / total_frames,
        "window_acc": correct_windows / total_windows,
        "precision": precision,
        "recall": recall,
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
    }

In [16]:
def train_one_epoch(
    model: CausalCNNGRU,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
    grad_clip: float,
    threshold: float,
    event_weight: float,
    freeze_except_last: bool = False,
    neg_window_weight: float = 2.0,
) -> dict[str, float]:
    if freeze_except_last:
        model.eval()
    else:
        model.train()
    total_loss = 0.0
    correct_frames = 0
    total_frames = 0
    correct_windows = 0
    total_windows = 0

    for batch in loader:
        imu = batch["imu"].to(device)
        frame_labels = batch["frame_labels"].to(device)
        window_label = batch["window_label"].to(device)
        imu = _augment_positive_windows(imu, window_label)

        logits, _ = model(imu)
        loss = frame_soft_ce(
            logits,
            frame_labels,
            event_weight,
            window_label=window_label,
            neg_window_weight=neg_window_weight,
        )
        probs = torch.softmax(logits, dim=-1)

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()

        total_loss += loss.item() * imu.size(0)
        pred = probs.argmax(dim=-1)
        true_cls = frame_labels.argmax(dim=-1)
        correct_frames += (pred == true_cls).sum().item()
        total_frames += true_cls.numel()

        window_pred = window_class_from_probs(probs, threshold)
        window_true = batch["window_label"].to(device)
        correct_windows += (window_pred == window_true).sum().item()
        total_windows += imu.size(0)

    n = len(loader.dataset)
    return {
        "loss": total_loss / n,
        "frame_acc": correct_frames / total_frames,
        "window_acc": correct_windows / total_windows,
    }

In [17]:
history = []

for epoch in range(1, cfg.training.epochs + 1):
    train_metrics = train_one_epoch(
        model,
        train_loader,
        optimizer,
        device,
        cfg.training.grad_clip,
        cfg.training.prediction_threshold,
        cfg.training.event_loss_weight,
        freeze_except_last=cfg.freeze_except_last,
        neg_window_weight=cfg.training.neg_window_weight,
    )
    val_metrics = evaluate(
        model,
        val_loader,
        device,
        cfg.training.prediction_threshold,
        cfg.training.event_loss_weight,
        neg_window_weight=cfg.training.neg_window_weight,
    )


    scheduler.step()

    print(
        f"Epoch {epoch:3d}/{cfg.training.epochs} | "
        f"train loss={train_metrics['loss']:.4f} acc={train_metrics['window_acc']:.3f} | "
        f"val loss={val_metrics['loss']:.4f} acc={val_metrics['window_acc']:.3f} "
        f"P={val_metrics['precision']:.3f} R={val_metrics['recall']:.3f}"
    )

    if val_metrics["window_acc"] > best_val_acc:
        best_val_acc = val_metrics["window_acc"]
        torch.save(
            {
                "model_state": model.state_dict(),
                "model_config": model_kwargs,
                "train_config": cfg.to_dict(),
                "epoch": epoch,
                "val_metrics": val_metrics,
            },
            out_dir / "best.pt",
        )

    history.append({
        "epoch": epoch,
        **{f"train_{key}": value for key, value in train_metrics.items()},
        **{f"val_{key}": value for key, value in val_metrics.items()},
    })

    if early_stop.step(val_metrics):
        stopped_epoch = epoch
        print(
            f"Early stopping at epoch {epoch} "
            f"(no {cfg_es.monitor} improvement for {cfg_es.patience} epochs)"
        )
        break

torch.save(
    {
        "model_state": model.state_dict(),
        "model_config": model_kwargs,
        "train_config": cfg.to_dict(),
        "epoch": stopped_epoch,
    },
    out_dir / "last.pt",
)
print(f"Done. Best val window acc: {best_val_acc:.3f} (stopped at epoch {stopped_epoch})")
print(f"Checkpoints saved to {out_dir}/")

pd.DataFrame(history).to_csv(out_dir / 'history.csv', index=False)


Epoch   1/50 | train loss=0.8446 acc=0.371 | val loss=0.2775 acc=0.636 P=0.000 R=0.000
Epoch   2/50 | train loss=0.4083 acc=0.371 | val loss=0.2262 acc=0.750 P=1.000 R=0.312
Epoch   3/50 | train loss=0.3573 acc=0.547 | val loss=0.1707 acc=0.877 P=0.896 R=0.750
Epoch   4/50 | train loss=0.2911 acc=0.804 | val loss=0.1522 acc=0.800 P=0.645 R=1.000
Epoch   5/50 | train loss=0.2371 acc=0.877 | val loss=0.1307 acc=0.882 P=0.755 R=1.000
Epoch   6/50 | train loss=0.2181 acc=0.909 | val loss=0.1239 acc=0.905 P=0.792 R=1.000
Epoch   7/50 | train loss=0.2084 acc=0.917 | val loss=0.1304 acc=0.877 P=0.748 R=1.000
Epoch   8/50 | train loss=0.2084 acc=0.930 | val loss=0.1373 acc=0.827 P=0.678 R=1.000
Epoch   9/50 | train loss=0.1997 acc=0.939 | val loss=0.1282 acc=0.882 P=0.755 R=1.000
Epoch  10/50 | train loss=0.1976 acc=0.947 | val loss=0.1270 acc=0.805 P=0.650 R=1.000
Epoch  11/50 | train loss=0.1977 acc=0.937 | val loss=0.1254 acc=0.864 P=0.727 R=1.000
Epoch  12/50 | train loss=0.1888 acc=0.970 